# 01b. Make the dataset English

The Olist reviews are written in Portuguese. This step translates every distinct review text once, with a free local model, and writes `data/english/olist_order_reviews_dataset.csv`. From here on the whole pipeline (Bronze, embeddings, index, search, UI, benchmark) works on English text.

## What to look at

- The **live examples**: Portuguese in, English out, translated in front of you by the real model.
- The **guards table**: what the code does around the model, and why (the model alone drops sentences and mangles ALL-CAPS and emoji-only text).
- The **status** cell: how many rows and distinct texts are translated.
- The full job takes about 30 to 40 minutes the first time (it is resumable). It is OFF by default here. Run `make translate` in a terminal, or set `NB_RUN_TRANSLATE = True`.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
import pandas as pd
from data_source import locate_csvs, ORDERS, REVIEWS
from translation import translate_texts, split_sentences, HF_MODEL
_, reviews_csv, source = locate_csvs(prefer_english=False)
print("data source:", source, "|", reviews_csv)
print("translation model:", HF_MODEL, "(CTranslate2 int8, CPU)")

In [ ]:
# Portuguese examples picked at random from the real reviews (Olist only), plus two hand-picked tricky ones
if source == "olist":
    raw = pd.read_csv(reviews_csv, dtype=str, keep_default_na=False)["review_comment_message"]
    raw = raw[raw.str.len().between(15, 160)].sample(8, random_state=7).tolist()
else:
    raw = []   # synthetic data is already English
raw += ["RAPIDEZ E EFICIÊNCIA.", "esta faltando 1 unidade"]
en, rejected = translate_texts(raw)
pd.DataFrame({"Portuguese": raw, "English": en})

In [ ]:
pd.DataFrame([
    ("long review", "split into sentences and translated piece by piece (the model drops sentences from long inputs)"),
    ("ALL CAPS", "lower-cased first (the model does not translate shouted text)"),
    ("\"........\" / \"!!!!!\"", "runs of the same symbol collapsed to three"),
    ("emoji or digits only", "no letters, so kept as is"),
    ("empty or runaway output", "rejected, original text kept (counted and logged)"),
    ("no final punctuation", "a period is added before and removed after (short inputs otherwise get junk appended)"),
], columns=["input", "what the code does"])

In [ ]:
print("sentence split of a long review:")
for s in split_sentences("Comprei o produto no dia 03. Chegou antes do prazo, muito bem embalado. Porém veio sem a nota fiscal e ninguém me respondeu."):
    print("  -", s)

In [ ]:
NB_RUN_TRANSLATE = False   # True = run the whole job here (~30-40 min first time, resumable)
if NB_RUN_TRANSLATE:
    run_script("01b_translate_dataset.py")
else:
    print("skipped: run `make translate` in a terminal, or set NB_RUN_TRANSLATE = True")

In [ ]:
english = common.DATA_DIR / "english" / REVIEWS
if english.exists():
    d = pd.read_csv(english, dtype=str, keep_default_na=False)
    print(f"{english.relative_to(common.ROOT)}: {len(d):,} rows")
    d[d.review_comment_message_pt.str.len() > 60][["review_comment_message_pt", "review_comment_message"]].sample(5, random_state=1).rename(
        columns={"review_comment_message_pt": "original (Portuguese)", "review_comment_message": "dataset now (English)"})
else:
    print("English dataset not built yet. Run `make translate`.")

## Honest limits

This is machine translation of short, slangy, misspelled reviews. Most reviews come out fine; some come out odd (a misspelling like *otino* can become a wrong word). For semantic search that is fine, because the embedding model captures the overall meaning, but do not treat any single translated sentence as exact. The Portuguese originals are kept in the `_pt` columns of the CSV.

We compared four free models on real reviews: the two Marian `opus-mt` models loop or produce unrelated text on short reviews, NLLB-200 invents sentences, and `unicamp-dl/translation-pt-en-t5` stayed closest to the source. CTranslate2 int8 makes it about 4x faster than plain PyTorch on a CPU.